In [ ]:
from lsst.rsp import RSPDiscovery
from astropy.table import Table
discovery = RSPDiscovery("prompt")
service = discovery.get_tap_client()
assert service is not None

In [ ]:
query = """
SELECT TOP 10 diaObjectId, ra, dec
FROM ppdb.DiaObject
"""
results = service.run_async(query)
assert len(results) > 0

In [ ]:
ut1 = results.to_table()
ut1["radius"] = 0.001
ut1

In [ ]:
query = """
SELECT u.diaObjectId AS upload_id, o.diaObjectId, o.ra, o.dec,
      DISTANCE(POINT('ICRS', o.ra, o.dec), POINT('ICRS', u.ra, u.dec)) AS dist
FROM ppdb.DiaObject AS o
JOIN TAP_UPLOAD.ut1 AS u
  ON DISTANCE(POINT('ICRS', o.ra, o.dec),
              POINT('ICRS', u.ra, u.dec)) < u.radius
"""
job = service.submit_job(query, uploads={"ut1": ut1})
job.run()
job.wait(phases=['COMPLETED', 'ERROR'], timeout=300)
print('Job phase is', job.phase)
print('Job ID is', job.job.jobid)
if job.phase == 'ERROR':
    job.raise_if_error()
results = job.fetch_result().to_table()
assert len(results) == len(ut1)
assert (results['upload_id'] == results['diaObjectId']).all()
print(results)